# Data Collection and Processing

In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import re

### Load the data from zipfile

In [2]:
import zipfile
with zipfile.ZipFile('../raw_data/tweet-bot-data.zip', 'r') as z:
    print(z.namelist())

['tweet-bot-data/', 'tweet-bot-data/bot/', 'tweet-bot-data/bot/tweets.csv', 'tweet-bot-data/bot/users.csv', 'tweet-bot-data/real/', 'tweet-bot-data/real/tweets.csv', 'tweet-bot-data/real/users.csv']


In [3]:
with zipfile.ZipFile('../raw_data/tweet-bot-data.zip', 'r') as z:
    with z.open('tweet-bot-data/real/users.csv') as f:
        real_users = pd.read_csv(f, encoding='latin-1')
    with z.open('tweet-bot-data/bot/users.csv') as f:
        bot_users = pd.read_csv(f, encoding='latin-1')
    with z.open('tweet-bot-data/real/tweets.csv') as f:
        real_tweets = pd.read_csv(f, encoding='latin-1')
    with z.open('tweet-bot-data/bot/tweets.csv') as f:
        bot_tweets = pd.read_csv(f, encoding='latin-1')

/tmp/ipykernel_170022/598530355.py:7: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  real_tweets = pd.read_csv(f, encoding='latin-1')
/tmp/ipykernel_170022/598530355.py:9: DtypeWarning: Columns (8,11) have mixed types. Specify dtype option on import or set low_memory=False.
  bot_tweets = pd.read_csv(f, encoding='latin-1')


### Create relational database tables from original CSVs

In [ ]:
# Add bot labels 
real_users['bot_label'] = 0
bot_users['bot_label'] = 1
real_tweets['bot_label'] = 0
bot_tweets['bot_label'] = 1

# Combine tables
users_raw = pd.concat([real_users, bot_users], ignore_index=True)
tweets_raw = pd.concat([real_tweets, bot_tweets], ignore_index=True)

# Make table 1: users 
user_cols = ['id', 'name', 'screen_name', 'statuses_count', 'followers_count',
             'friends_count', 'favourites_count', 'listed_count', 'verified',
             'location', 'lang', 'created_at', 'description', 'bot_label']

users = users_raw[user_cols].drop_duplicates(subset='id').copy()
users.rename(columns={'id': 'user_id'}, inplace=True)

# Make table 2: tweets
tweet_cols = ['id', 'user_id', 'text', 'retweet_count', 'reply_count',
              'favorite_count', 'num_hashtags', 'num_urls', 'num_mentions', 'created_at']

tweets = tweets_raw[tweet_cols].drop_duplicates(subset='id').copy()
tweets.rename(columns={'id': 'tweet_id'}, inplace=True)

# Make table 3: hashtags 
hashtags = (tweets[['tweet_id', 'text']]
    .assign(hashtag=tweets['text'].str.findall(r'#\w+'))
    .explode('hashtag')
    .dropna(subset=['hashtag'])
    .reset_index(drop=True)[['tweet_id', 'hashtag']])
hashtags['hashtag'] = hashtags['hashtag'].str.lower()
hashtags.insert(0, 'hashtag_id', range(1, len(hashtags) + 1))

# Make table 4: locations 
locations = (users[['location']]
    .drop_duplicates()
    .dropna()
    .reset_index(drop=True))
locations.insert(0, 'location_id', range(1, len(locations) + 1))

# Add location_id back to users
users = users.merge(locations, on='location', how='left')

print("users:", users.shape)
print("tweets:", tweets.shape)
print("hashtags:", hashtags.shape)
print("locations:", locations.shape)

users: (6825, 15)
tweets: (3035389, 10)
hashtags: (711200, 3)
locations: (3602, 2)


In [ ]:
# Drop rows where tweet_id is not numeric
tweets = tweets[pd.to_numeric(tweets['tweet_id'], errors='coerce').notna()]
tweets['tweet_id'] = tweets['tweet_id'].astype('int64')

# Save to parquet files with fastparquet
users.to_parquet('../processed_data/users.parquet', engine='fastparquet', index=False)
tweets.to_parquet('../processed_data/tweets.parquet', engine='fastparquet', index=False)
hashtags.to_parquet('../processed_data/hashtags.parquet', engine='fastparquet', index=False)
locations.to_parquet('../processed_data/locations.parquet', engine='fastparquet', index=False)

In [ ]:
users.columns
tweets.columns
hashtags.columns
locations.columns